# Spark Architecture & Setup

Apache Spark is a unified analytics engine for large-scale data processing. It provides in-memory computation, which makes it significantly faster than disk-based systems like classic Hadoop MapReduce. Spark supports batch processing, streaming, machine learning, and graph computation through a single, consistent API available in Python (PySpark), Scala, Java, and R.

In this notebook we will:
- Understand the key components of a Spark cluster
- Connect a SparkSession to a running standalone cluster
- Run our very first distributed computation

## Spark Cluster Architecture

Spark follows a **master/worker** architecture. Every Spark application has one **Driver** process and one or more **Executor** processes spread across worker nodes.

```
┌─────────────────────────────────────────────────────────────┐
│                        Driver Program                        │
│                                                             │
│   SparkSession  ──►  SparkContext  ──►  DAG Scheduler       │
└──────────────────────────┬──────────────────────────────────┘
                           │  submits tasks
                           ▼
              ┌────────────────────────┐
              │    Cluster Manager     │
              │  (Standalone / YARN /  │
              │   Mesos / Kubernetes)  │
              └──────────┬─────────────┘
                         │  allocates resources
            ┌────────────┼────────────┐
            ▼            ▼            ▼
      ┌──────────┐ ┌──────────┐ ┌──────────┐
      │  Worker  │ │  Worker  │ │  Worker  │
      │  Node 1  │ │  Node 2  │ │  Node 3  │
      │          │ │          │ │          │
      │ Executor │ │ Executor │ │ Executor │
      │ (Tasks)  │ │ (Tasks)  │ │ (Tasks)  │
      └──────────┘ └──────────┘ └──────────┘
```

**Key components:**

| Component | Role |
|-----------|------|
| **Driver** | Runs your `main()` / notebook code; orchestrates the job |
| **SparkContext** | Entry point for low-level RDD API; created inside SparkSession |
| **Cluster Manager** | Negotiates resources (cores, memory) across the cluster |
| **Worker / Executor** | JVM process on each node that actually runs tasks and stores data |

> **Tip:** In this environment we use Spark's built-in **Standalone** cluster manager. The master URL `spark://spark-master:7077` tells the driver where to find the cluster manager.

In [1]:
from pyspark.sql import SparkSession

# SparkSession is the single entry point for all Spark functionality.
# .master()  -> tells Spark WHERE to run (our standalone cluster)
# .appName() -> label that appears in the Spark Web UI
# .getOrCreate() -> returns an existing session if one already exists
spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")   # standalone cluster master
    .appName("01-Architecture")            # name shown in Spark UI
    .getOrCreate()
)

print("SparkSession created successfully.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 13:21:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession created successfully.


In [3]:
# Inspect the session to verify we are connected to the cluster

print("Spark version          :", spark.version)
print("Master URL             :", spark.sparkContext.master)
print("Default parallelism    :", spark.sparkContext.defaultParallelism)
print("Application ID         :", spark.sparkContext.applicationId)

Spark version          : 3.5.3
Master URL             : spark://spark-master:7077
Default parallelism    : 4
Application ID         : app-20260624132141-0000


## Understanding the Output

- **`spark.version`** — confirms you are running Spark 3.5.x.
- **`spark.sparkContext.master`** — the `spark://spark-master:7077` URL confirms the driver has successfully registered with the standalone cluster manager (not running locally in pseudo-distributed mode).
- **`spark.sparkContext.defaultParallelism`** — the default number of partitions Spark will create for operations that don't specify a partition count explicitly. For a standalone cluster this equals the **total number of CPU cores available across all workers**. More cores → more partitions → more parallel tasks.
- **`applicationId`** — a unique ID assigned by the cluster manager; you can use it to find this job in the Spark Web UI at `http://spark-master:8080`.

In [4]:
# Access the underlying SparkContext (the older, lower-level API)
sc = spark.sparkContext

# sc.parallelize() distributes a local Python collection across the cluster,
# creating an RDD (Resilient Distributed Dataset).
rdd = sc.parallelize(range(1, 11))   # numbers 1 through 10

# .collect() is an ACTION — it pulls all partition data back to the driver.
# This is what actually triggers execution on the cluster.
result = rdd.collect()

print("RDD contents:", result)
print("Number of partitions:", rdd.getNumPartitions())

RDD contents: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Number of partitions: 4


## What Just Happened?

Even though `[1, 2, 3, ..., 10]` looks trivial, several distributed steps occurred:

1. **`sc.parallelize(range(1,11))`** — the driver sliced the range into partitions and recorded a *logical plan* (no data moved yet — this is **lazy evaluation**).
2. **`rdd.collect()`** — this *action* told Spark to execute the plan:
   - The **DAG Scheduler** converted the logical plan into **stages**.
   - Each stage was split into **tasks** (one per partition).
   - The **Task Scheduler** shipped tasks to **Executors** on the worker nodes.
   - Executors ran the tasks and sent results back to the driver.
3. The driver reassembled the partition results into a single Python list.

> **Key insight:** Transformations (`.map`, `.filter`, etc.) are *lazy* — they only build a plan. Actions (`.collect`, `.count`, `.save`) are *eager* — they trigger real computation. We will explore this in depth in the next notebook.

In [5]:
# Always stop the SparkSession when you are done.
# This releases executor resources back to the cluster.
spark.stop()
print("SparkSession stopped. Resources released.")

SparkSession stopped. Resources released.
